# Hand off a curated S3 prefix to another team with KMS, locked down to read-only, audited in CloudTrail

It is Friday afternoon. The analytics lead just pinged: "We are setting up the BI account next week, we need read access to the `curated/v2/` prefix by Monday standup. Full encryption, audit trail to satisfy finance. Can you have it ready?"

You own the platform's raw S3 lake. The analytics team needs one prefix, encrypted with a key you control, with proof in CloudTrail that they can read what they are supposed to read and get denied everywhere else. You are going to simulate the cross-account boundary inside a single AWS account using two IAM roles (one producer, one consumer). The STS AssumeRole pattern is identical to a real cross-account handoff; the only thing missing is a second billing account.

The deliverables are concrete. The consumer role can `GetObject` and decrypt under `curated/v2/`. The consumer role gets `AccessDenied` on `raw/`. The consumer role gets `AccessDenied` if it tries to decrypt anything outside the curated encryption context. CloudTrail logs both the success and the denials, attributable to the consumer role.

Four checkpoints. Four authorization layers stacked: IAM identity policy, S3 bucket policy, KMS key policy, KMS encryption context.

**Time.** Plan for about 60 minutes hands-on. The CloudTrail data events checkpoint (Checkpoint 4) is the slowest: trail events take 5 to 15 minutes to flush, and the validator polls with a 20-minute ceiling. Budget 90 minutes total if you hit any KMS key-policy debugging.

**Cost.** About 5 cents for a clean run. The KMS customer-managed key is the most expensive thing in the lab at $1 per month, prorated to roughly 4 cents for one hour. CloudTrail management events are free; data events are $0.10 per 100k and this lab generates well under 50. The cleanup cell schedules the KMS key for deletion with the AWS-minimum 7-day window; the orphan scan recognizes pending-deletion keys and does not block your next run.


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.8.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT section 12.

import atexit
import base64
import getpass
import json
import os
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "cross-account-data-handoff-kms"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"  # this lab runs in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource identifiers. The S3 bucket and trail bucket get the account ID suffix
# because S3 bucket names are global. Other names are scoped by the lab slug.
PRODUCER_ROLE = f"labrun-{LAB_ID}-producer-role"
CONSUMER_ROLE = f"labrun-{LAB_ID}-consumer-role"
KEY_ALIAS = f"alias/labrun-{LAB_ID}"
TRAIL_NAME = f"labrun-{LAB_ID}-trail"
INLINE_POLICY_NAME = f"labrun-{LAB_ID}-consumer-curated-access"

# Resolved in the next cell once we know the account ID.
BUCKET_NAME = None
TRAIL_BUCKET = None
KEY_ID = None
KEY_ARN = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything, per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"This lab runs in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
TRAIL_BUCKET = f"labrun-{LAB_ID}-trail-{ACCOUNT_ID}"
print(f"Lake bucket resolved: {BUCKET_NAME}")
print(f"Trail bucket resolved: {TRAIL_BUCKET}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Reverse-creation order. No critical (hourly-billed) resources in this lab.
# The KMS key is the highest-cost item and it prorates to a few cents per
# session; the cleanup cell schedules it for deletion with the AWS-minimum
# 7-day window via the kms_key adapter type.
#
# Note on adapter coverage: CloudTrail trails are not currently supported as
# a labrun-checks CleanupResource type, so trail teardown happens manually
# inside the cleanup cell before run_cleanup() is called. The trail S3
# bucket and the main lake bucket are then both reachable via the
# s3_bucket adapter as usual.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="s3_bucket",
        id=TRAIL_BUCKET,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{TRAIL_BUCKET} --force",
    ),
    CleanupResource(
        type="kms_alias",
        id=KEY_ALIAS,
        region=REGION,
        cli_delete_command=f"aws kms delete-alias --alias-name {KEY_ALIAS}",
    ),
    CleanupResource(
        type="kms_key",
        id="",  # populated after the key is created in the scaffold cell
        region=REGION,
        cli_delete_command="aws kms schedule-key-deletion --key-id <key-id> --pending-window-in-days 7",
    ),
    CleanupResource(
        type="iam_role",
        id=CONSUMER_ROLE,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {CONSUMER_ROLE}",
    ),
    CleanupResource(
        type="iam_role",
        id=PRODUCER_ROLE,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {PRODUCER_ROLE}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    Errors are swallowed because atexit handlers must not raise; the
    dedicated cleanup cell is the authoritative cleanup entry point.
    """
    try:
        try:
            ct = boto3.client(
                "cloudtrail",
                aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
                aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
                aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
                region_name=REGION,
            )
            try:
                ct.stop_logging(Name=TRAIL_NAME)
            except ClientError:
                pass
            try:
                ct.delete_trail(Name=TRAIL_NAME)
            except ClientError:
                pass
        except Exception:
            pass
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell blocks if any resources
    tagged labrun:lab-slug=<slug> exist. Pending-deletion KMS keys also
    surface here; if you see a KMS arn in the orphan list with a recent
    deletion-pending state, AWS will tear it down automatically when the
    7-day window elapses and the next run is fine.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))

    kms_client = boto3.client(
        "kms",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    real_orphans = []
    for arn in arns:
        if ":kms:" in arn and "key/" in arn:
            key_id = arn.split("key/")[-1]
            try:
                meta = kms_client.describe_key(KeyId=key_id).get("KeyMetadata", {})
                if meta.get("KeyState") == "PendingDeletion":
                    continue
            except ClientError:
                pass
        real_orphans.append(arn)
    return real_orphans


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources.")
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("remove them manually with the AWS CLI commands the cleanup cell")
    print("prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Setup: scaffolding the lake, the roles, the key, and the trail

Before the four tasks begin, you need infrastructure in place. This cell creates the things the lab assumes already exist on the platform team's side: the lake bucket, the producer and consumer IAM roles, the KMS key plus alias, a seed `raw/sample.parquet` object (so Checkpoint 3 has something to deny against), the CloudTrail trail and its log bucket (with management events only).

Read it before running. The KMS key policy here is intentionally permissive for the lab: it allows the AWS account root principal and grants both the producer and consumer roles the standard set of KMS actions. Tasks 2 and 3 are where you tighten that policy to scope the consumer's grants to the curated prefix only. Cell output prints the KMS key ARN so the task cells can reference it.


In [ ]:
# NBVAL_SKIP
# Scaffold the lake, the two roles, the KMS key, and the CloudTrail trail.
# This cell is platform-side setup, not pedagogy. The KMS key policy here
# is permissive on purpose: the lab tasks (Task 2 and Task 3) are where
# the consumer's access gets scoped down.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
kms = boto3.client(
    "kms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ct = boto3.client(
    "cloudtrail",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# 1. Lake bucket.
s3.create_bucket(Bucket=BUCKET_NAME)
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
s3.put_object(Bucket=BUCKET_NAME, Key="raw/sample.parquet", Body=b"raw-seed-bytes")
print(f"Lake bucket created and seeded: {BUCKET_NAME}")

# 2. Producer + consumer roles. Trust policy is account-root so the same
# notebook session can assume them via STS.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
        "Action": "sts:AssumeRole",
    }],
}
for role_name in (PRODUCER_ROLE, CONSUMER_ROLE):
    try:
        iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"labrun {LAB_ID} {role_name}",
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        print(f"Role created: {role_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            print(f"Role {role_name} already exists, reusing.")
        else:
            raise

producer_inline = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:PutObject", "s3:GetObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        },
        {
            "Effect": "Allow",
            "Action": ["kms:GenerateDataKey", "kms:Encrypt", "kms:Decrypt", "kms:DescribeKey"],
            "Resource": "*",
        },
    ],
}
iam.put_role_policy(
    RoleName=PRODUCER_ROLE,
    PolicyName=f"labrun-{LAB_ID}-producer-access",
    PolicyDocument=json.dumps(producer_inline),
)
print(f"Producer inline policy attached.")

# 3. KMS key with a permissive lab policy. Task 2 and Task 3 are where the
# consumer's access gets scoped.
permissive_key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "EnableRoot",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "kms:*",
            "Resource": "*",
        },
        {
            "Sid": "AllowProducer",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:role/{PRODUCER_ROLE}"},
            "Action": [
                "kms:GenerateDataKey",
                "kms:Encrypt",
                "kms:Decrypt",
                "kms:DescribeKey",
            ],
            "Resource": "*",
        },
    ],
}
key_resp = kms.create_key(
    Description=f"labrun {LAB_ID} lab KMS key",
    KeyUsage="ENCRYPT_DECRYPT",
    Origin="AWS_KMS",
    Policy=json.dumps(permissive_key_policy),
    Tags=[{"TagKey": LAB_TAG_KEY, "TagValue": LAB_TAG_VALUE}],
)
KEY_ID = key_resp["KeyMetadata"]["KeyId"]
KEY_ARN = key_resp["KeyMetadata"]["Arn"]
for r in CLEANUP_MANIFEST:
    if r.type == "kms_key":
        r.id = KEY_ID
        r.cli_delete_command = (
            f"aws kms schedule-key-deletion --key-id {KEY_ID} --pending-window-in-days 7"
        )
        break

try:
    kms.create_alias(AliasName=KEY_ALIAS, TargetKeyId=KEY_ID)
    print(f"KMS key created: {KEY_ARN}")
    print(f"KMS alias created: {KEY_ALIAS}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"KMS alias {KEY_ALIAS} already exists.")
    else:
        raise

# 4. Trail bucket. CloudTrail requires a bucket policy that lets the
# CloudTrail service principal write log objects to it.
s3.create_bucket(Bucket=TRAIL_BUCKET)
s3.put_bucket_tagging(
    Bucket=TRAIL_BUCKET,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
trail_bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AWSCloudTrailAclCheck",
            "Effect": "Allow",
            "Principal": {"Service": "cloudtrail.amazonaws.com"},
            "Action": "s3:GetBucketAcl",
            "Resource": f"arn:aws:s3:::{TRAIL_BUCKET}",
        },
        {
            "Sid": "AWSCloudTrailWrite",
            "Effect": "Allow",
            "Principal": {"Service": "cloudtrail.amazonaws.com"},
            "Action": "s3:PutObject",
            "Resource": f"arn:aws:s3:::{TRAIL_BUCKET}/AWSLogs/{ACCOUNT_ID}/*",
            "Condition": {
                "StringEquals": {"s3:x-amz-acl": "bucket-owner-full-control"}
            },
        },
    ],
}
s3.put_bucket_policy(Bucket=TRAIL_BUCKET, Policy=json.dumps(trail_bucket_policy))
print(f"Trail bucket created and policy attached: {TRAIL_BUCKET}")

# 5. CloudTrail trail. Management events on; data events are added by the
# student in Task 4.
try:
    ct.create_trail(
        Name=TRAIL_NAME,
        S3BucketName=TRAIL_BUCKET,
        IncludeGlobalServiceEvents=False,
        IsMultiRegionTrail=False,
        EnableLogFileValidation=False,
        TagsList=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    ct.start_logging(Name=TRAIL_NAME)
    print(f"CloudTrail trail created and started: {TRAIL_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "TrailAlreadyExistsException":
        ct.start_logging(Name=TRAIL_NAME)
        print(f"Trail {TRAIL_NAME} already exists, started logging.")
    else:
        raise

# IAM propagation. STS AssumeRole is stricter than other IAM-dependent calls.
print()
print("Scaffold complete. Letting IAM and KMS propagate...")
time.sleep(15)
print("Propagation buffer elapsed.")
print()
print(f"KEY_ID = {KEY_ID}")
print(f"KEY_ARN = {KEY_ARN}")

## Task 1: Producer writes two encrypted objects to `curated/v2/`

The platform team owns this side of the handoff. The analytics team will not see any object unless it lands under `curated/v2/` with the lab KMS key referenced as the `SSEKMSKeyId`. Default `put_object` does not encrypt with KMS; you have to ask for it.

Write two small parquet placeholders (`sample1.parquet` and `sample2.parquet`) to `curated/v2/`. The bytes can be anything; this lab is about the encryption headers, not the file contents. Each upload must set:

- `ServerSideEncryption='aws:kms'`
- `SSEKMSKeyId=KEY_ARN`

If you skip either, `head_object` later reports `AES256` or no encryption header, and Checkpoint 1 fails. The producer role's IAM policy and the KMS key policy already grant `kms:GenerateDataKey`, so the put calls will succeed once the parameters are right.


In [ ]:
# NBVAL_SKIP
# Task 1: Write two encrypted objects to curated/v2/ using the lab KMS key.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

curated_objects = [
    ("curated/v2/sample1.parquet", b"curated-payload-one"),
    ("curated/v2/sample2.parquet", b"curated-payload-two"),
]

for key, body in curated_objects:
    # YOUR CODE: Call s3.put_object() with Bucket=BUCKET_NAME, Key=key, Body=body,
    # ServerSideEncryption='aws:kms', and SSEKMSKeyId=KEY_ARN.
    print(f"Uploaded s3://{BUCKET_NAME}/{key}")

print()
print(f"Two encrypted objects landed under s3://{BUCKET_NAME}/curated/v2/")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: curated/v2/ has at least two objects, each encrypted with the
# lab KMS key.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            listing = s3_client.list_objects_v2(
                Bucket=BUCKET_NAME, Prefix="curated/v2/"
            )
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))

        contents = listing.get("Contents", [])
        if len(contents) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Found {len(contents)} object(s) under "
                    f"s3://{BUCKET_NAME}/curated/v2/; need at least 2."
                ),
            )

        bad = []
        for obj in contents:
            head = s3_client.head_object(Bucket=BUCKET_NAME, Key=obj["Key"])
            sse = head.get("ServerSideEncryption")
            sse_kms_key = head.get("SSEKMSKeyId")
            if sse != "aws:kms":
                bad.append(
                    f"{obj['Key']}: ServerSideEncryption is {sse!r}, expected 'aws:kms'"
                )
                continue
            if not sse_kms_key or KEY_ID not in sse_kms_key:
                bad.append(
                    f"{obj['Key']}: SSEKMSKeyId is {sse_kms_key!r}, expected the lab key {KEY_ID!r}"
                )
        if bad:
            return CheckpointResult(
                status="fail",
                error_reason="; ".join(bad),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The objects are landing, but `head_object` says they are either unencrypted or encrypted with the default AES256 key. By default, `put_object` does not use KMS unless you tell it to. Two extra parameters do the work.

</details>

<details><summary>Hint 2 (direction)</summary>

The two parameters are `ServerSideEncryption` and `SSEKMSKeyId`. Set `ServerSideEncryption='aws:kms'` to switch from S3-managed AES256 to KMS, and `SSEKMSKeyId=KEY_ARN` to point S3 at the lab key (the variable is already defined from the scaffold cell). Both have to be on every `put_object` call to a curated object.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

curated_objects = [
    ("curated/v2/sample1.parquet", b"curated-payload-one"),
    ("curated/v2/sample2.parquet", b"curated-payload-two"),
]

for key, body in curated_objects:
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=key,
        Body=body,
        ServerSideEncryption="aws:kms",
        SSEKMSKeyId=KEY_ARN,
    )
    print(f"Uploaded s3://{BUCKET_NAME}/{key}")

print()
print(f"Two encrypted objects landed under s3://{BUCKET_NAME}/curated/v2/")
```

</details>


**Wallet check.** Two `kms:GenerateDataKey` calls (one per object) at $0.03 per 10,000 requests. Two `s3:PutObject` calls inside the free request quota. Running total: under a tenth of a cent. Your morning coffee cost about a thousand times more.


## Task 2: Grant the consumer role read + decrypt on `curated/v2/`

The analytics role exists but has no permissions yet. You need two things lined up before `s3:GetObject` will return bytes:

1. **Consumer IAM identity policy** that allows `s3:GetObject` on `arn:aws:s3:::{BUCKET}/curated/v2/*` and `kms:Decrypt` on the lab key ARN. The IAM side authorizes the consumer to call S3 and KMS.
2. **KMS key policy** that names the consumer role ARN as a Principal with `kms:Decrypt` permission. Without this, the consumer's IAM `kms:Decrypt` is dead on arrival: KMS evaluates the key policy first, and a key policy that does not name the consumer principal blocks the call before IAM ever runs.

This is the most common confusion in cross-account KMS: granting `kms:Decrypt` in the consumer's IAM is necessary but not sufficient. The key policy is the ground truth.

Build it in this order:
1. Attach an inline policy to the consumer role with `s3:GetObject` + `kms:Decrypt` scoped to `curated/v2/*` and the lab key ARN.
2. Replace the KMS key policy with a version that keeps the root-allow and producer-allow statements and adds a new `AllowConsumerDecrypt` statement for the consumer role.

The cell ends with a 15-second propagation pause; STS AssumeRole + first decrypt against a freshly-updated key policy can race the propagation window otherwise.


In [ ]:
# NBVAL_SKIP
# Task 2: Attach an inline policy to the consumer role, then update the KMS
# key policy to include an AllowConsumerDecrypt statement.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
kms = boto3.client(
    "kms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

consumer_inline = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "s3:GetObject",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/curated/v2/*",
        },
        {
            "Effect": "Allow",
            "Action": "kms:Decrypt",
            "Resource": KEY_ARN,
        },
    ],
}
# YOUR CODE: Attach consumer_inline to CONSUMER_ROLE using iam.put_role_policy()
# with PolicyName=INLINE_POLICY_NAME and PolicyDocument=json.dumps(consumer_inline).
print(f"Inline policy attached to {CONSUMER_ROLE}.")

updated_key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "EnableRoot",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "kms:*",
            "Resource": "*",
        },
        {
            "Sid": "AllowProducer",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:role/{PRODUCER_ROLE}"},
            "Action": [
                "kms:GenerateDataKey",
                "kms:Encrypt",
                "kms:Decrypt",
                "kms:DescribeKey",
            ],
            "Resource": "*",
        },
        {
            "Sid": "AllowConsumerDecrypt",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}"},
            "Action": "kms:Decrypt",
            "Resource": "*",
        },
    ],
}
# YOUR CODE: Apply updated_key_policy using kms.put_key_policy() with
# KeyId=KEY_ID, PolicyName="default", and Policy=json.dumps(updated_key_policy).
print(f"KMS key policy updated with AllowConsumerDecrypt for {CONSUMER_ROLE}.")

print()
print("IAM and KMS policy changes applied. Letting AWS propagate...")
time.sleep(15)
print("Propagation buffer elapsed. The consumer role should be able to read curated/v2/ now.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Assume the consumer role with STS, build a new s3 client from
# the temporary credentials, confirm GetObject + KMS decrypt works on
# curated/v2/sample1.parquet, and confirm PutObject is denied.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        sts_client = boto3.client(
            "sts",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}"
        try:
            assume_resp = sts_client.assume_role(
                RoleArn=role_arn,
                RoleSessionName="labrun-checkpoint-2",
            )
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not assume {role_arn}. STS returned "
                    f"{e.response['Error']['Code']}: {e.response['Error']['Message']}."
                ),
            )

        temp_creds = assume_resp["Credentials"]
        s3_assumed = boto3.client(
            "s3",
            aws_access_key_id=temp_creds["AccessKeyId"],
            aws_secret_access_key=temp_creds["SecretAccessKey"],
            aws_session_token=temp_creds["SessionToken"],
            region_name=REGION,
        )

        try:
            obj = s3_assumed.get_object(
                Bucket=BUCKET_NAME, Key="curated/v2/sample1.parquet"
            )
            _ = obj["Body"].read()
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Consumer role could not read s3://{BUCKET_NAME}/curated/v2/sample1.parquet. "
                    f"Got {code_str}: {e.response['Error']['Message']}. "
                    f"If the error is KMS-related, check the key policy AllowConsumerDecrypt statement; "
                    f"if S3, check the consumer's inline IAM policy Resource."
                ),
            )

        try:
            s3_assumed.put_object(
                Bucket=BUCKET_NAME,
                Key="curated/v2/poison.parquet",
                Body=b"should-not-land",
                ServerSideEncryption="aws:kms",
                SSEKMSKeyId=KEY_ARN,
            )
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Consumer role successfully wrote curated/v2/poison.parquet. "
                    f"The inline policy is too broad; PutObject should not be allowed."
                ),
            )
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str != "AccessDenied":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Expected AccessDenied on PutObject, got {code_str}."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

If the consumer's GetObject is failing on the S3 side, the inline IAM policy is the place to look. If GetObject works but the response body raises a KMS error, the IAM is fine and the KMS key policy is missing the consumer principal. Two different layers; two different fixes.

</details>

<details><summary>Hint 2 (direction)</summary>

The inline policy on the consumer role needs `s3:GetObject` on `arn:aws:s3:::{BUCKET_NAME}/curated/v2/*` and `kms:Decrypt` on `KEY_ARN`. The KMS key policy needs a new Statement with the consumer's role ARN (`arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}`) as `Principal` and `kms:Decrypt` as the Action. Both API calls are `put_role_policy` and `put_key_policy`; you are replacing existing policy documents, not appending to them.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
kms = boto3.client(
    "kms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

consumer_inline = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "s3:GetObject",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/curated/v2/*",
        },
        {
            "Effect": "Allow",
            "Action": "kms:Decrypt",
            "Resource": KEY_ARN,
        },
    ],
}
iam.put_role_policy(
    RoleName=CONSUMER_ROLE,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(consumer_inline),
)
print(f"Inline policy attached to {CONSUMER_ROLE}.")

updated_key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "EnableRoot",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "kms:*",
            "Resource": "*",
        },
        {
            "Sid": "AllowProducer",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:role/{PRODUCER_ROLE}"},
            "Action": [
                "kms:GenerateDataKey",
                "kms:Encrypt",
                "kms:Decrypt",
                "kms:DescribeKey",
            ],
            "Resource": "*",
        },
        {
            "Sid": "AllowConsumerDecrypt",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}"},
            "Action": "kms:Decrypt",
            "Resource": "*",
        },
    ],
}
kms.put_key_policy(
    KeyId=KEY_ID,
    PolicyName="default",
    Policy=json.dumps(updated_key_policy),
)
print(f"KMS key policy updated with AllowConsumerDecrypt for {CONSUMER_ROLE}.")

print()
print("IAM and KMS policy changes applied. Letting AWS propagate...")
time.sleep(15)
print("Propagation buffer elapsed. The consumer role should be able to read curated/v2/ now.")
```

</details>


**Wallet check.** STS AssumeRole calls are free. One `kms:Decrypt` per object (a fraction of a cent). The KMS key continues to prorate at about 4 cents per session-hour. Running total: still under one cent of API spend. The lab is cheaper than a parking meter.


## Task 3: Close the negative space (deny `raw/` and out-of-context decrypts)

Task 2 unlocked the curated prefix. That is the easy half. The audit deliverable also needs proof that the consumer is denied everywhere it should be denied. Two attack surfaces to close:

1. **Bucket policy must deny GetObject on `raw/`** for the consumer role. The consumer's IAM policy already does not allow it, but a future engineer might widen the inline policy by accident. An explicit Deny on `raw/*` in the bucket policy is the defense-in-depth backstop. Wildcard actions (`s3:*`, `*`) are acceptable per the blueprint's policy-validation rules; an explicit `s3:GetObject` Deny is just as good.
2. **KMS key policy must require encryption context for the consumer's Decrypt grant.** Right now `AllowConsumerDecrypt` is universal: if the consumer ever got hold of a ciphertext blob encrypted with this key from some other workflow, they could decrypt it. The fix is a `Condition` block on the key policy statement that pins `kms:EncryptionContext:aws:s3:arn` to the curated prefix object ARN pattern. Outside that context, `Decrypt` is denied.

Build it:
1. Write a bucket policy with a Deny statement naming the consumer role as Principal and `s3:GetObject` (or `s3:*` wildcard) on `arn:aws:s3:::{BUCKET_NAME}/raw/*`. Apply via `s3.put_bucket_policy`.
2. Replace the `AllowConsumerDecrypt` statement on the KMS key policy with a version that adds a `Condition` block: `"StringLike": {"kms:EncryptionContext:aws:s3:arn": "arn:aws:s3:::{BUCKET_NAME}/curated/v2/*"}`.

After this task, the consumer is locked in: read curated, deny raw, decrypt only in-context.


In [ ]:
# NBVAL_SKIP
# Task 3: Bucket policy denies consumer on raw/*. KMS key policy adds an
# encryption-context Condition on the consumer Decrypt grant.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
kms = boto3.client(
    "kms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

consumer_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}"
bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "DenyConsumerRawRead",
            "Effect": "Deny",
            "Principal": {"AWS": consumer_role_arn},
            "Action": "s3:GetObject",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/raw/*",
        },
    ],
}
# YOUR CODE: Apply bucket_policy via s3.put_bucket_policy() with
# Bucket=BUCKET_NAME and Policy=json.dumps(bucket_policy).
print(f"Bucket policy applied: consumer Deny on s3://{BUCKET_NAME}/raw/*")

context_arn_prefix = f"arn:aws:s3:::{BUCKET_NAME}/curated/v2/*"
scoped_key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "EnableRoot",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "kms:*",
            "Resource": "*",
        },
        {
            "Sid": "AllowProducer",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:role/{PRODUCER_ROLE}"},
            "Action": [
                "kms:GenerateDataKey",
                "kms:Encrypt",
                "kms:Decrypt",
                "kms:DescribeKey",
            ],
            "Resource": "*",
        },
        {
            "Sid": "AllowConsumerDecryptScoped",
            "Effect": "Allow",
            "Principal": {"AWS": consumer_role_arn},
            "Action": "kms:Decrypt",
            "Resource": "*",
            "Condition": {
                "StringLike": {
                    "kms:EncryptionContext:aws:s3:arn": context_arn_prefix
                }
            },
        },
    ],
}
# YOUR CODE: Apply scoped_key_policy via kms.put_key_policy() with KeyId=KEY_ID,
# PolicyName="default", Policy=json.dumps(scoped_key_policy).
print(f"KMS key policy updated: consumer Decrypt scoped via aws:s3:arn context.")

print()
print("Policy changes applied. Letting AWS propagate...")
time.sleep(10)
print("Propagation buffer elapsed.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Consumer role gets AccessDenied on raw/sample.parquet
# and on a kms.decrypt() call against a ciphertext produced outside the
# curated encryption context.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        sts_client = boto3.client(
            "sts",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}"
        try:
            assume_resp = sts_client.assume_role(
                RoleArn=role_arn,
                RoleSessionName="labrun-checkpoint-3",
            )
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not assume {role_arn}. STS returned "
                    f"{e.response['Error']['Code']}: {e.response['Error']['Message']}."
                ),
            )

        temp_creds = assume_resp["Credentials"]
        s3_assumed = boto3.client(
            "s3",
            aws_access_key_id=temp_creds["AccessKeyId"],
            aws_secret_access_key=temp_creds["SecretAccessKey"],
            aws_session_token=temp_creds["SessionToken"],
            region_name=REGION,
        )
        kms_assumed = boto3.client(
            "kms",
            aws_access_key_id=temp_creds["AccessKeyId"],
            aws_secret_access_key=temp_creds["SecretAccessKey"],
            aws_session_token=temp_creds["SessionToken"],
            region_name=REGION,
        )

        try:
            s3_assumed.get_object(Bucket=BUCKET_NAME, Key="raw/sample.parquet")
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Consumer role successfully read s3://{BUCKET_NAME}/raw/sample.parquet. "
                    f"The bucket policy is not denying access to raw/."
                ),
            )
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str != "AccessDenied":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Expected AccessDenied on raw/sample.parquet, got {code_str}."
                    ),
                )

        base_kms = boto3.client(
            "kms",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            ct_resp = base_kms.encrypt(
                KeyId=KEY_ID,
                Plaintext=b"out-of-context-payload",
                EncryptionContext={"workflow": "out-of-context-probe"},
            )
        except ClientError as e:
            return CheckpointResult(
                status="error",
                error_reason=(
                    f"Could not encrypt the probe payload with base creds: {e}. "
                    f"This is an environmental issue, not a student bug."
                ),
            )

        try:
            kms_assumed.decrypt(
                CiphertextBlob=ct_resp["CiphertextBlob"],
                EncryptionContext={"workflow": "out-of-context-probe"},
            )
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Consumer role successfully decrypted a ciphertext produced outside "
                    f"the curated encryption context. The AllowConsumerDecrypt key policy "
                    f"statement needs a Condition on kms:EncryptionContext:aws:s3:arn."
                ),
            )
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str != "AccessDenied":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Expected AccessDenied on the out-of-context decrypt, got {code_str}. "
                        f"Verify the Condition block in the AllowConsumerDecrypt key policy statement."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two attack surfaces here, two different policies. If the consumer can still read `raw/`, the bucket policy is the missing or wrong piece. If the consumer can decrypt anything the key has ever encrypted, the KMS key policy's consumer statement is too broad: it has no `Condition` block.

</details>

<details><summary>Hint 2 (direction)</summary>

Bucket policy needs a `Deny` statement with the consumer role ARN as Principal, `s3:GetObject` as Action, and `arn:aws:s3:::{BUCKET_NAME}/raw/*` as Resource. The KMS key policy needs the existing `AllowConsumerDecrypt` statement, but with an added `Condition` block of `"StringLike": {"kms:EncryptionContext:aws:s3:arn": "arn:aws:s3:::{BUCKET_NAME}/curated/v2/*"}`. When S3 calls `kms:Decrypt` for an object under `curated/v2/`, it automatically passes the object ARN as encryption context, and the condition matches; for anything else, the condition fails and the decrypt is denied.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
kms = boto3.client(
    "kms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

consumer_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}"
bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "DenyConsumerRawRead",
            "Effect": "Deny",
            "Principal": {"AWS": consumer_role_arn},
            "Action": "s3:GetObject",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/raw/*",
        },
    ],
}
s3.put_bucket_policy(Bucket=BUCKET_NAME, Policy=json.dumps(bucket_policy))
print(f"Bucket policy applied: consumer Deny on s3://{BUCKET_NAME}/raw/*")

context_arn_prefix = f"arn:aws:s3:::{BUCKET_NAME}/curated/v2/*"
scoped_key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "EnableRoot",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "kms:*",
            "Resource": "*",
        },
        {
            "Sid": "AllowProducer",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:role/{PRODUCER_ROLE}"},
            "Action": [
                "kms:GenerateDataKey",
                "kms:Encrypt",
                "kms:Decrypt",
                "kms:DescribeKey",
            ],
            "Resource": "*",
        },
        {
            "Sid": "AllowConsumerDecryptScoped",
            "Effect": "Allow",
            "Principal": {"AWS": consumer_role_arn},
            "Action": "kms:Decrypt",
            "Resource": "*",
            "Condition": {
                "StringLike": {
                    "kms:EncryptionContext:aws:s3:arn": context_arn_prefix
                }
            },
        },
    ],
}
kms.put_key_policy(
    KeyId=KEY_ID,
    PolicyName="default",
    Policy=json.dumps(scoped_key_policy),
)
print(f"KMS key policy updated: consumer Decrypt scoped via aws:s3:arn context.")

print()
print("Policy changes applied. Letting AWS propagate...")
time.sleep(10)
print("Propagation buffer elapsed.")
```

</details>


**Wallet check.** Bucket policy update is free. KMS key policy update is free. One additional `kms:Encrypt` + one `kms:Decrypt` from the validator probe at $0.03 per 10,000. The meter sits comfortably under two cents. You could run this checkpoint thirty more times before the lab cost a dime.


## Task 4: Enable CloudTrail data events and confirm the audit trail

The trail is logging management events from the scaffold cell, but management events do not capture S3 GetObject calls. Finance asked for proof of who read what; that lives in CloudTrail data events. Two pieces of work:

1. Configure the trail with an event selector that includes data events for the lake bucket. The boto3 API is `cloudtrail.put_event_selectors` with an `EventSelectors` list containing one entry whose `DataResources` includes `Type='AWS::S3::Object'` and `Values=['arn:aws:s3:::{BUCKET_NAME}/']`. The trailing slash matters: it scopes the data events to objects inside that bucket.
2. Re-trigger the audit events. Assume the consumer role one more time and make three calls: one successful `GetObject` on `curated/v2/sample1.parquet`, one denied `GetObject` on `raw/sample.parquet`, and one denied `PutObject` on `curated/v2/poison.parquet`. The validator will look for these in CloudTrail.

CloudTrail data events lag 5 to 15 minutes between API call and trail flush. The validator polls every 30 seconds for up to 20 minutes with quirky wait messages. Plan accordingly: do not start Task 4 if you have less than 20 minutes of session window left.


In [ ]:
# NBVAL_SKIP
# Task 4: Enable CloudTrail data events for the lake bucket, then re-trigger
# the audit events (one success, two denials) so the trail has something to
# log.

ct = boto3.client(
    "cloudtrail",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

event_selectors = [
    {
        "ReadWriteType": "All",
        "IncludeManagementEvents": True,
        "DataResources": [
            {
                "Type": "AWS::S3::Object",
                "Values": [f"arn:aws:s3:::{BUCKET_NAME}/"],
            }
        ],
    }
]
# YOUR CODE: Apply event_selectors via ct.put_event_selectors() with
# TrailName=TRAIL_NAME and EventSelectors=event_selectors.
print(f"CloudTrail data events configured on {TRAIL_NAME} for s3://{BUCKET_NAME}/")

assume_resp = sts.assume_role(
    RoleArn=f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}",
    RoleSessionName="labrun-audit-trigger",
)
temp_creds = assume_resp["Credentials"]
s3_assumed = boto3.client(
    "s3",
    aws_access_key_id=temp_creds["AccessKeyId"],
    aws_secret_access_key=temp_creds["SecretAccessKey"],
    aws_session_token=temp_creds["SessionToken"],
    region_name=REGION,
)

try:
    s3_assumed.get_object(Bucket=BUCKET_NAME, Key="curated/v2/sample1.parquet")
    print("Triggered: GetObject on curated/v2/sample1.parquet (expected success).")
except ClientError as e:
    print(f"Unexpected error on curated GetObject: {e.response['Error']['Code']}")

try:
    s3_assumed.get_object(Bucket=BUCKET_NAME, Key="raw/sample.parquet")
    print("Unexpected: raw GetObject succeeded.")
except ClientError as e:
    print(f"Triggered: GetObject on raw/sample.parquet (expected AccessDenied, got {e.response['Error']['Code']}).")

try:
    s3_assumed.put_object(
        Bucket=BUCKET_NAME,
        Key="curated/v2/poison.parquet",
        Body=b"never-lands",
        ServerSideEncryption="aws:kms",
        SSEKMSKeyId=KEY_ARN,
    )
    print("Unexpected: curated PutObject succeeded.")
except ClientError as e:
    print(f"Triggered: PutObject on curated/v2/poison.parquet (expected AccessDenied, got {e.response['Error']['Code']}).")

print()
print("Audit events fired. CloudTrail data-event flushing takes 5-15 minutes.")
print("Run the next cell when you are ready; the validator will poll for the events.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Poll CloudTrail lookup_events for events attributable to the
# consumer role: at least one Successful GetObject and at least two
# AccessDenied events. Ceiling 20 minutes. Quirky wait messages every poll.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        ct_client = boto3.client(
            "cloudtrail",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            sel = ct_client.get_event_selectors(TrailName=TRAIL_NAME)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not read event selectors on {TRAIL_NAME}: "
                    f"{e.response['Error']['Code']}. Did Task 4's put_event_selectors run?"
                ),
            )
        has_s3_data = False
        for s in sel.get("EventSelectors", []):
            for dr in s.get("DataResources", []):
                if dr.get("Type") == "AWS::S3::Object":
                    values = dr.get("Values", [])
                    if any(BUCKET_NAME in v for v in values):
                        has_s3_data = True
                        break
        if not has_s3_data:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Trail {TRAIL_NAME} is not configured with S3 data events for {BUCKET_NAME}. "
                    f"Call cloudtrail.put_event_selectors with a DataResources entry of type "
                    f"AWS::S3::Object pointing at arn:aws:s3:::{BUCKET_NAME}/."
                ),
            )

        wait_messages = [
            "CloudTrail flushes data events on its own schedule. About 5 to 15 minutes. Reading the latest...",
            "Asking CloudTrail nicely if your events landed yet...",
            "The audit-trail bus stops every five minutes. Waiting for the next one...",
            "Tailing the trail bucket. The flush is slow but it does happen...",
            "Squinting at the CloudTrail console on your behalf...",
        ]
        deadline = time.time() + 20 * 60
        attempt = 0
        success_found = False
        denied_count = 0

        while time.time() < deadline:
            attempt += 1
            msg = wait_messages[attempt % len(wait_messages)]
            print(f"[poll {attempt}] {msg}")
            events_collected = []
            try:
                paginator = ct_client.get_paginator("lookup_events")
                for page in paginator.paginate(
                    LookupAttributes=[
                        {"AttributeKey": "EventSource", "AttributeValue": "s3.amazonaws.com"}
                    ],
                    MaxResults=50,
                ):
                    events_collected.extend(page.get("Events", []))
            except ClientError as e:
                return CheckpointResult(
                    status="error",
                    error_reason=f"lookup_events failed: {e}",
                )

            success_found = False
            denied_count = 0
            for ev in events_collected:
                raw = ev.get("CloudTrailEvent")
                if not raw:
                    continue
                try:
                    payload = json.loads(raw)
                except Exception:
                    continue
                ui = payload.get("userIdentity", {})
                session_ctx = ui.get("sessionContext", {})
                session_issuer = session_ctx.get("sessionIssuer", {})
                issuer_name = session_issuer.get("userName")
                if issuer_name != CONSUMER_ROLE:
                    continue
                event_name = payload.get("eventName") or ev.get("EventName")
                error_code = payload.get("errorCode")
                if event_name == "GetObject" and not error_code:
                    success_found = True
                if error_code == "AccessDenied":
                    denied_count += 1

            print(
                f"          attributable to {CONSUMER_ROLE}: "
                f"successful GetObject={'yes' if success_found else 'no'}, "
                f"AccessDenied count={denied_count}"
            )

            if success_found and denied_count >= 2:
                return CheckpointResult(status="pass")

            time.sleep(30)

        return CheckpointResult(
            status="fail",
            error_reason=(
                f"After 20 minutes of polling, did not find both a Successful GetObject "
                f"and at least 2 AccessDenied events attributable to {CONSUMER_ROLE} "
                f"(found success={success_found}, denied_count={denied_count}). "
                f"Confirm Task 4 re-triggered the three S3 calls AFTER put_event_selectors ran."
            ),
        )
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If the validator times out at 20 minutes, two questions: is the trail actually configured with S3 data events, and were the three test calls fired AFTER the event selector was applied? CloudTrail does not back-fill: only events emitted after the data-events selector lands get logged.

</details>

<details><summary>Hint 2 (direction)</summary>

`cloudtrail.put_event_selectors` configures data events on the existing trail. The relevant API surface: `TrailName=TRAIL_NAME` and an `EventSelectors` list. Inside the selector, `DataResources` is the list that turns on per-resource event logging; for S3, `Type="AWS::S3::Object"` and `Values=["arn:aws:s3:::{BUCKET_NAME}/"]` (note the trailing slash, which AWS treats as "any object in this bucket"). After this call lands, the three S3 calls (one success, one raw deny, one curated put deny) need to fire so the trail has something to log.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
ct = boto3.client(
    "cloudtrail",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

event_selectors = [
    {
        "ReadWriteType": "All",
        "IncludeManagementEvents": True,
        "DataResources": [
            {
                "Type": "AWS::S3::Object",
                "Values": [f"arn:aws:s3:::{BUCKET_NAME}/"],
            }
        ],
    }
]
ct.put_event_selectors(TrailName=TRAIL_NAME, EventSelectors=event_selectors)
print(f"CloudTrail data events configured on {TRAIL_NAME} for s3://{BUCKET_NAME}/")

assume_resp = sts.assume_role(
    RoleArn=f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}",
    RoleSessionName="labrun-audit-trigger",
)
temp_creds = assume_resp["Credentials"]
s3_assumed = boto3.client(
    "s3",
    aws_access_key_id=temp_creds["AccessKeyId"],
    aws_secret_access_key=temp_creds["SecretAccessKey"],
    aws_session_token=temp_creds["SessionToken"],
    region_name=REGION,
)

try:
    s3_assumed.get_object(Bucket=BUCKET_NAME, Key="curated/v2/sample1.parquet")
    print("Triggered: GetObject on curated/v2/sample1.parquet (expected success).")
except ClientError as e:
    print(f"Unexpected error on curated GetObject: {e.response['Error']['Code']}")

try:
    s3_assumed.get_object(Bucket=BUCKET_NAME, Key="raw/sample.parquet")
    print("Unexpected: raw GetObject succeeded.")
except ClientError as e:
    print(f"Triggered: GetObject on raw/sample.parquet (expected AccessDenied, got {e.response['Error']['Code']}).")

try:
    s3_assumed.put_object(
        Bucket=BUCKET_NAME,
        Key="curated/v2/poison.parquet",
        Body=b"never-lands",
        ServerSideEncryption="aws:kms",
        SSEKMSKeyId=KEY_ARN,
    )
    print("Unexpected: curated PutObject succeeded.")
except ClientError as e:
    print(f"Triggered: PutObject on curated/v2/poison.parquet (expected AccessDenied, got {e.response['Error']['Code']}).")

print()
print("Audit events fired. CloudTrail data-event flushing takes 5-15 minutes.")
```

</details>


**Wallet check.** CloudTrail management events are free. Data events cost $0.10 per 100,000; this lab generated maybe 10 to 20 events total, so the trail spend is effectively zero. The KMS key continues to prorate. Lab session total so far: about 5 cents. Less than a vending-machine candy bar.


In [ ]:
# NBVAL_SKIP
# Cleanup. Trail teardown happens manually first so stop-logging completes
# before the bucket policy is removed below. labrun-checks 0.8.0 does ship
# a cloudtrail_trail adapter (stop-logging prep is built in), so this
# manual step is duplicate-safe and could be folded into CLEANUP_MANIFEST
# in a future revision. Then run_cleanup() handles everything else via the
# AWS adapter in reverse-creation order. KMS key is the only resource that
# lingers: AWS requires a 7-day minimum pending-deletion window for KMS
# keys; the orphan scan in setup recognizes that state and does not block
# your next run.

import sys

try:
    ct = boto3.client(
        "cloudtrail",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    try:
        ct.stop_logging(Name=TRAIL_NAME)
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("TrailNotFoundException",):
            print(f"Warning on stop_logging: {e.response['Error']['Code']}")
    try:
        ct.delete_trail(Name=TRAIL_NAME)
        print(f"CloudTrail trail deleted: {TRAIL_NAME}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "TrailNotFoundException":
            print(f"CloudTrail trail {TRAIL_NAME} already gone.")
        else:
            print(f"Failed to delete trail {TRAIL_NAME}: {e}")
except Exception as e:
    print(f"Trail teardown raised: {e}")

# Remove the bucket policy so the s3_bucket adapter can list+delete the
# bucket without 403s from a leftover Deny statement.
try:
    s3_cleanup = boto3.client(
        "s3",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    try:
        s3_cleanup.delete_bucket_policy(Bucket=BUCKET_NAME)
    except ClientError:
        pass
except Exception:
    pass

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

# A KMS key in PendingDeletion state is not a real warning: AWS schedules it
# for hard delete on the 7-day timer. Filter those warnings before counting
# failures so a clean run does not look dirty just because the key is in
# its pending window.
real_warnings = []
for w in result.warnings:
    if "kms_key" in w and ("PendingDeletion" in w or (KEY_ID and KEY_ID in w)):
        continue
    real_warnings.append(w)

failed_ids: set[str] = set()
for w in real_warnings:
    for r in CLEANUP_MANIFEST:
        if r.id and r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status if not failed_count else "dirty")

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.05.** The KMS key carried most of the spend, prorated to roughly 4 cents for one session-hour. A handful of `kms:GenerateDataKey`, `kms:Encrypt`, and `kms:Decrypt` calls at $0.03 per 10,000 added another fraction of a cent. CloudTrail data events for this lab volume rounded to zero. The bucket and roles cost nothing. The KMS key is scheduled for hard deletion in 7 days (AWS minimum window); your bill stops accruing the moment that delete schedule lands.


## These are not graded. They are for you.

Three questions worth sitting with before you ship this pattern in production.

1. Walk through what happens when the consumer's `get_object` call lands at S3 for an object encrypted with the lab KMS key. Name each authorization layer in order: IAM identity policy on the consumer, S3 bucket policy, KMS key policy, KMS encryption context. Which one is evaluated first, which one is the most common place an engineer's mental model goes wrong, and what symptom does each layer produce on failure? Map at least one specific `botocore.exceptions.ClientError` code per layer.

2. The trail in this lab logged management events from the moment it was created, but data events did not appear until Task 4 added the event selector. If you were on call and got a finance request asking for "every consumer-role GetObject in the last 30 days," and you only had management-event logs, what is your actual answer? Is "we did not log it, it does not exist" acceptable, and what does the production fix look like (multi-region trail, organization trail, S3 server access logs, or all of the above)?

3. This lab simulated cross-account inside one account using two IAM roles. In a real two-account handoff, what specifically changes? Walk through (a) the trust policy on the consumer role, (b) the KMS key policy `Principal` field, (c) the bucket policy `Principal` field, (d) the `aws:PrincipalAccount` or `aws:SourceAccount` Conditions you would add, and (e) the CloudTrail account that captures the events. Knowing the simulation's tradeoffs is the difference between an exam answer and a production design.

## Exam-Style Practice

**Q1.** An analytics role can `GetObject` on a KMS-encrypted S3 object via the bucket policy and an inline IAM policy that grants `kms:Decrypt` on the key ARN, but the call raises `KMS.AccessDeniedException` on the underlying decrypt step. The most likely cause is:

A) The inline IAM policy's `kms:Decrypt` Resource is too narrow and does not cover the data-key sub-resource.

B) The KMS key's resource (key) policy does not list the analytics role as a Principal with `kms:Decrypt`.

C) The bucket policy is missing an `s3:GetObjectAttributes` Allow for the analytics role.

D) CloudTrail is not configured for data events on the bucket.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because IAM `kms:Decrypt` on the key ARN is sufficient at the identity layer; KMS does not require sub-resource granularity for symmetric customer-managed keys.
- B) Right because KMS evaluates the key policy independently of the caller's IAM policy. If the key policy does not name the analytics role as a Principal, IAM `kms:Decrypt` is dead on arrival. This is the most common confusion when teams onboard a new consumer to a KMS-encrypted bucket.
- C) Wrong because `s3:GetObjectAttributes` is metadata-only and not on the GetObject critical path. S3 returns `KMS.AccessDeniedException`, not a missing-permission error, when the IAM is fine and KMS denies.
- D) Wrong because CloudTrail is an observability layer, not an authorization layer. Whether trail logging is on or off has no bearing on whether KMS Decrypt succeeds.

</details>

**Q2.** The analytics team needs read-only access to one S3 prefix in your account, and you must be able to revoke that access in 90 days when the project ends. The bucket is KMS-encrypted with a key your team owns. Which mechanism is the best fit?

A) Apply a bucket policy with `NotPrincipal` listing every account except the analytics account.

B) Configure a Lake Formation column-level grant on a Glue table that maps to the prefix.

C) Provision IAM Identity Center permission sets and assign them to the analytics team.

D) Create an S3 Access Point with a time-bounded bucket policy that names the analytics account, plus a KMS key grant scoped to that access point's ARN.

<details><summary>Show answer</summary>

**D**.

- A) Wrong because `NotPrincipal` is fragile and tends to over-grant in subtle ways; AWS itself recommends against it for cross-account scenarios. Enumerating every account is also unmanageable at scale.
- B) Wrong because Lake Formation grants are scoped to Glue catalog objects, not S3 prefixes directly. If the analytics team is reading raw S3 (not Athena/Glue queries), Lake Formation is not the right plane.
- C) Wrong because IAM Identity Center is for human-user federation across multiple accounts, not for cross-account programmatic data access. It does not solve the "revoke at 90 days" problem cleanly either.
- D) Right because S3 Access Points scope cross-account access cleanly to a specific prefix surface, and the access-point policy is independent of the bucket policy (which keeps the production policy clean). Pairing the access point with a scoped KMS grant matches the dual-control encryption pattern the lab just built. Revoking is one access-point delete plus one KMS grant revoke.

</details>

**Q3.** Your CloudTrail trail logs management events. Finance asks for a list of every `s3:GetObject` call against the curated/ prefix in the last 24 hours by the analytics role. You run `cloudtrail.lookup_events` filtered by `EventSource=s3.amazonaws.com` and get zero rows. What is the diagnostic?

A) The trail's S3 bucket has not flushed its log files yet; wait another 15 minutes and re-run the lookup.

B) `lookup_events` only returns the last 90 days; 24 hours of GetObject calls is below the API's resolution.

C) The trail does not have an event selector configured for S3 data events; management events alone do not capture per-object S3 GetObject.

D) The analytics role's IAM policy is denying CloudTrail introspection; switch to a role with `cloudtrail:LookupEvents` permission.

<details><summary>Show answer</summary>

**C**.

- A) Wrong because trail flushes are 5 to 15 minutes; 24 hours is well past any flush window. The events simply do not exist.
- B) Wrong because `lookup_events` does scan up to 90 days but has no minimum resolution; the issue is the events are not being captured at all.
- C) Right because per-object S3 GetObject (and PutObject, DeleteObject) are CloudTrail data events, not management events. Without an event selector containing a `DataResources` entry of type `AWS::S3::Object` pointing at the bucket, the trail captures nothing at the object level. This is the lesson Task 4 of the lab encoded.
- D) Wrong because `cloudtrail:LookupEvents` permission affects the caller's ability to query, not whether events were captured. If the call had failed on permission, it would have raised AccessDenied; an empty result instead means the events were not logged.

</details>
